In [4]:
import numpy as np
import cv2
import time
import os
from PIL import Image

cap = cv2.VideoCapture(0)
cap.set(3, 640)  #宽640
cap.set(4, 480)  #高480

font                   = cv2.FONT_HERSHEY_SIMPLEX   #文本字体
bottomLeftCornerOfText = (10, 25)  #文本显示的位置
fontScale              = 1       #字体大小
fontColor              = (255,255,255)  #字体颜色白色
lineType               = 2

print('Input image class index: ', end = '')
class_index = input()

print('Input maximum image number of class {}: '.format(class_index), end = '')
max_image_num = input()

model_bin = "./models/models/face_detector/opencv_face_detector_uint8.pb"
config_text = "./models/models/face_detector/opencv_face_detector.pbtxt";

net_face = cv2.dnn.readNetFromTensorflow(model_bin, config=config_text)


folder_path = os.path.join('./face_dataset/train', str(class_index))
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

i = 0
while(True):
    ret, frame = cap.read()
    h, w, c = frame.shape
    #将图像转换为神经网络的输入格式
    blobImage = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0), False, False) 
    #将处理后的图像输入到人脸检测网络中，并获取网络的输出结果
    net_face.setInput(blobImage)
    cvOut = net_face.forward()
   # 检测人脸位置
    for detection in cvOut[0,0,:,:]:
        score = float(detection[2])
        objIndex = int(detection[1])  #检测到的对象类型
        if score > 0.5:
            left = int(detection[3]*w) #边界框左边界的相对x坐标（范围0-1）后同理
            top = int(detection[4]*h)
            right = int(detection[5]*w)
            bottom = int(detection[6]*h)
        
        #构建保存图像的文件路径，从原始帧中裁剪出人脸区域，并整为240x320像素的大小，保存到指定路径···
            saved_file_path = os.path.join(folder_path, 'image_' + str(i) + '.png')
            pic = frame[top:bottom,left:right]
            resized = cv2.resize(pic,(240,320))
            cv2.imwrite(saved_file_path,resized)

        #在原始图像上绘制一个蓝色矩形框（BGR中(255,0,0)是蓝色），标记检测到的人脸位置。
            img = cv2.rectangle(frame,(left, top), (right, bottom),(255, 0, 0), 2)
            cv2.putText(frame,'Image: ' + str(i), 
                bottomLeftCornerOfText, 
                font, 
                fontScale,
                fontColor,
                lineType)

    cv2.imshow('frame', frame)
    cv2.waitKey(1)
    i+=1

    if cv2.waitKey(1) == 27 or i >= int(max_image_num):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)  #mac用户需要添加

Input image class index: Input maximum image number of class 3: 

-1